In [ ]:
import numpy as np
from itertools import chain
import pandas as pd
import json

ages_table_path = 'ages_table.csv'
ages_table = pd.read_csv(ages_table_path)

**Selecting subjects test**

In [ ]:
# Selecting subjects test

range_groups = np.array([[0, 1], [1, 12], [25, 40], [40, 65], [65, 74]])
np.random.seed(7)
test_percent = 0.15
# select 10% subjects of each group to test
subjects_test = []

for interval in range_groups:
    
    start, end = interval
    subjects = ages_table.loc[(ages_table['ages'] >= start) & (ages_table['ages'] < end), "codes"].tolist()
    quantity = int(test_percent * len(subjects))
    files_test = list(np.random.choice(subjects, size=quantity, replace=False))
    subjects_test.append(files_test)

subjects_test = list(chain.from_iterable(subjects_test))
print(len(subjects_test))

23


**Define train subjects**

In [ ]:
# Define train subjects
# create a dictionary of groups by ages
dict_ages = {}

for age in ages_table['ages'].unique():
    codes_list = ages_table.loc[ages_table['ages'] == age, "codes"].tolist()
    codes_list = [code for code in codes_list if code not in subjects_test]
    if not codes_list:
        continue
    else:
        dict_ages[age] = codes_list

with open("subjects_train.json", "w", encoding="utf-8") as f:
    json.dump(dict_ages, f, ensure_ascii=False, indent=4)

**Normalization parameters by age**

Note: let $x$ be age (years). For RR mean, RR standard deviation, and pNN50 use the formulas below.

RR mean:

$$
\text{RR\_mean} = 505 \cdot x^{0.122}
$$

For RR standard deviation and pNN50 use the age-dependent formulas:

If $x \le 12$:

$$
\text{RR\_std} = 80 \cdot x^{0.26} \\,
\text{pNN50} = 0.037 \cdot x^{0.78}
$$

If $x > 12$:

$$
\text{RR\_std} = 290 \cdot x^{-0.2} \\,
\text{pNN50} = 5 \cdot x^{-1.1}
$$

Python implementation:

```python
# x = age (years)
RR_mean = 505 * x**0.122
if x <= 12:
    RR_std = 80 * x**0.26
    pNN50 = 0.037 * x**0.78
else:
    RR_std = 290 * x**(-0.2)
    pNN50 = 5 * x**(-1.1)
```